In [8]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../models_code")))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))
import numpy as np
import pandas as pd
from svd import SVD

from sqlmodel import create_engine, Session, select
from models import UserRating, Movie, User

In [ ]:
# joining MovieLens ratings with Movie App ratings
def include_db_ratings():
    data = pd.read_csv('../../maintenance/data/ratings_cut.csv').drop('timestamp', axis=1)
    
    # adding database path, creating engine 
    real_db_path = os.path.abspath(os.path.join(os.getcwd(), "../../database.db"))
    sqlite_url = f"sqlite:///{real_db_path}"
    engine = create_engine(sqlite_url, connect_args={"check_same_thread": False})

    # getting ratings
    with Session(engine) as session:
        statement = (
            select(Movie.id, UserRating.user_id, UserRating.rating)
            .where(Movie.id == UserRating.movie_id)
        )
        ratings = session.exec(statement).all()
    ratings_from_db = pd.DataFrame(ratings, columns=["movieId", "OriginalUserId", "rating"])

    # scalling to 0-5 scale
    ratings_from_db['rating'] /= 2

    # mapping new user ids
    current_max_id = int(data['userId'].max())
    unique_db_users = ratings_from_db['OriginalUserId'].unique()
    user_mapping = {
        db_user_id: new_model_id 
        for new_model_id, db_user_id in enumerate(unique_db_users, start=current_max_id + 1)
    }
    ratings_from_db['userId'] = ratings_from_db['OriginalUserId'].map(user_mapping)

    # updating User.model_id in db
    with Session(engine) as session:
        for db_user_id, new_model_id in user_mapping.items():
            user = session.exec(select(User).where(User.id == int(db_user_id))).first()
            
            user.model_id = new_model_id
            session.add(user)

        session.commit()

    # joining all ratings
    ratings_from_db = ratings_from_db[["userId", "movieId", "rating"]]
    all_ratings = pd.concat([data, ratings_from_db], ignore_index=True)

    # OPTIONAL: checking if User.model_id was updated correctly
    # with Session(engine) as session:
    #     statement = (
    #         select(Movie.id, UserRating.user_id, UserRating.rating, User.model_id)
    #         .where(Movie.id == UserRating.movie_id)
    #         .where(UserRating.user_id == User.id)
    #     )
    #     results = session.exec(statement).all()
    
    return all_ratings

In [ ]:
all_ratings = include_db_ratings() # all available ratings

In [ ]:
# data
data = all_ratings.loc[:, ['userId', 'movieId', 'rating']].values

# hyperparameters
n_factors = 110
n_epochs = 48
lr_all = 0.007445
reg_all = 0.047003

svd = SVD(n_factors=n_factors, n_epochs=n_epochs, lr_all=lr_all, reg_all=reg_all)

In [ ]:
# model training
svd = svd.fit(data)


In [ ]:
# saving model
svd.save("../models/SVD.npz")

In [ ]:
# OPTIONAL: checking rmse, mae

# rmse, mae = svd.cross_validation(data, n_splits=3)
# print('rmse:', rmse)
# print('mae:', mae)